In [12]:
pip uninstall dquant

^C
Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install dquant==1.1.4

  Attempting uninstall: dquant
    Found existing installation: dquant 1.1.3.1
    Uninstalling dquant-1.1.3.1:
      Successfully uninstalled dquant-1.1.3.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Installation of dependencies

In [3]:
pip install dquant arch yfinance

  Using cached yfinance-1.3.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached multitasking-0.0.13-py3-none-any.whl.metadata (16 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.5-py3-none-any.whl.metadata (8.6 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached websockets-16.0-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached yfinance-1.3.0-py2.py3-none-any.whl (133 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached frozendict-2.4.7-py3-none-any.whl (16 kB)
Using cached multitasking-0.0.13-py3-none-any.whl (16 kB)
Using cached peewee-4.0.5-py3-none-any.whl (144 kB)
Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
Using cached websockets-16.0-cp312-cp312-win_amd64.whl


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
from sklearn.metrics import mean_absolute_error
from dquant.models import VolClustXGB
from arch import arch_model
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. ПАРАМЕТРЫ
# ============================================================
TICKERS = [
    'BTC-USD',    # Bitcoin
    'ETH-USD',    # Ethereum
    'EURUSD=X',   # EUR/USD
    'BZ=F',       # Brent Crude Oil
    'GC=F',       # Gold
    'SI=F',       # Silver
    'SPY'         # SPDR S&P 500 ETF
]
START_DATE = '2014-01-01'       # Начало всей выборки
SPLIT_DATE = '2023-01-01'       # Дата разделения train/test
END_DATE = '2024-12-31'         # Конец тестовой выборки

# DQuant
INPUT_BARS = 70                 # Сколько свечей подавать на вход
HORIZON = 10                     # Горизонт прогноза в днях
TREES_COUNT = 2000               # Максимальное число деревьев для early stopping
FEATURES = [                    # Список признаков (из документации)
    'returns'
]

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ И ПОДГОТОВКА
# ============================================================
def get_data(ticker, start_date, end_date):
    print(f"Загрузка {ticker} с {start_date} по {end_date}...")
    raw = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)
    # Приводим к формату DQuant: open, high, low, close, volume
    df = pd.DataFrame({
        'open':   raw[('Open', ticker)].values,
        'high':   raw[('High', ticker)].values,
        'low':    raw[('Low', ticker)].values,
        'close':  raw[('Close', ticker)].values,
        'volume': raw[('Volume', ticker)].values
    }, index=raw.index)
    return df

def parkinson_func(df):
  df = df.copy()
  df = df.iloc[1:]
  return np.array(np.sqrt(
        (1 / (4 * np.log(2))) * (np.log(df['high'] / df['low']))**2
    ))

def dquant_train():
    model_dq = VolClustXGB({}, early_stopping=True, output=False)
    model_dq.fit(
        df_train,
        feature_list=FEATURES,
        input_bars=INPUT_BARS,
        horizon=HORIZON,
        trees_count=TREES_COUNT,
        show_results=False,
        target_func=parkinson_func
    )
    return model_dq

# ============================================================
# 5. ПРОГНОЗЫ DQUANT НА ТЕСТОВОМ ПЕРИОДЕ
# ============================================================
def dquant_forecasting():
    model_dq = dquant_train()
    dquant_preds = []
    dquant_dates = []

    # Идём по каждому дню тестового периода, для которого возможен прогноз
    for i in range(INPUT_BARS, len(df_test)):
        # Текущая дата тестового дня
        test_date = df_test.index[i]

        # Берём все доступные данные до test_date (train + предыдущие дни test)
        available_data = df.loc[:test_date].iloc[-INPUT_BARS:].copy()
        if len(available_data) < INPUT_BARS:
            continue   # недостаточно истории

        try:
            forecast_vals = model_dq.forecast(available_data, show=False)
            # forecast_vals – массив длины HORIZON
            pred_vol = forecast_vals[-1] if HORIZON > 1 else forecast_vals[0]
        except Exception as e:
            print(f"Ошибка прогноза DQuant на {test_date.date()}: {e}")
            continue
    # Фактическое значение Parkinson vol на test_date
        actual_vol = df_test.loc[test_date, 'parkinson_vol']

        dquant_dates.append(test_date)
        dquant_preds.append({
            'date': test_date,
            'actual': actual_vol,
            'predicted': pred_vol
        })

    df_dquant = pd.DataFrame(dquant_preds).set_index('date')
    return df_dquant
# ============================================================
# 6. ПРОГНОЗЫ GARCH (РАСШИРЯЮЩЕЕСЯ ОКНО ДЛЯ ЧЕСТНОСТИ)
# ============================================================
def garch_forecasting():
    garch_preds = []
    garch_dates = []

    # Нам нужна история доходностей до каждого тестового дня
    returns_full = df['returns'].dropna() * 100   # масштабирование для GARCH

    for test_date in df_test.index:
        # Все доходности до test_date (включая train и прошлые дни test)
        hist_returns = returns_full[:test_date].iloc[:-1]  # не включаем сам test_date
        if len(hist_returns) < 500:
            continue   # слишком мало данных для обучения

        try:
            model_garch = arch_model(hist_returns, vol='GARCH', p=1, q=1, o=1, dist='normal')
            fitted = model_garch.fit(disp='off')
            forec = fitted.forecast(horizon=1, reindex=False)
            cond_var = forec.variance.values[-1, 0]
            pred_vol = np.sqrt(cond_var) / 100.0   # обратное масштабирование
        except Exception as e:
            print(f"Ошибка GARCH на {test_date.date()}: {e}")
            continue

        actual_vol = df.loc[test_date, 'parkinson_vol']
        garch_preds.append({
            'date': test_date,
            'actual': actual_vol,
            'predicted': pred_vol
        })

    df_garch = pd.DataFrame(garch_preds).set_index('date')
    return df_garch


def qlike(y_true, y_pred):
    sigma2_true = y_true**2
    sigma2_pred = np.maximum(y_pred**2, 1e-10)
    return np.mean(np.log(sigma2_pred) + sigma2_true / sigma2_pred)

for i in TICKERS:
    df = get_data(i, START_DATE, END_DATE)
    # Дневная доходность и Parkinson volatility (прокси реализованной волатильности)
    df['returns'] = df['close'].pct_change()
    df['parkinson_vol'] = np.sqrt(
        (1 / (4 * np.log(2))) * (np.log(df['high'] / df['low']))**2
    )
    df = df.dropna().copy()

    print(f"Всего свечей: {len(df)}")
    print(f"Период: {df.index[0].date()} – {df.index[-1].date()}")

    # ============================================================
    # 3. РАЗДЕЛЕНИЕ НА TRAIN И TEST (СТАТИЧНЫЙ СПЛИТ)
    # ============================================================
    train_mask = df.index < SPLIT_DATE
    test_mask = df.index >= SPLIT_DATE

    df_train = df[train_mask].copy()
    df_test = df[test_mask].copy()

    print(f"\nTrain: {len(df_train)} свечей (до {SPLIT_DATE})")
    print(f"Test:  {len(df_test)} свечей (с {SPLIT_DATE})")

    df_dquant = dquant_forecasting()
    df_garch = garch_forecasting()
    common_dates = df_dquant.index.intersection(df_garch.index)
    print(f"\nОбщих дат для сравнения: {len(common_dates)}")

    dquant_aligned = df_dquant.loc[common_dates]
    garch_aligned = df_garch.loc[common_dates]

    # Метрики
    mae_dq = mean_absolute_error(dquant_aligned['actual'], dquant_aligned['predicted'])
    mae_garch = mean_absolute_error(garch_aligned['actual'], garch_aligned['predicted'])

    qlike_dq = qlike(dquant_aligned['actual'], dquant_aligned['predicted'])
    qlike_garch = qlike(garch_aligned['actual'], garch_aligned['predicted'])

    # Тест Диболда-Мариано (на квадратах ошибок)
    err_dq = (dquant_aligned['actual'] - dquant_aligned['predicted'])**2
    err_garch = (garch_aligned['actual'] - garch_aligned['predicted'])**2
    d = err_dq - err_garch
    n = len(d)
    # Простая реализация без учёта автокорреляции (h=1)
    dm_stat = np.mean(d) / np.std(d, ddof=1) * np.sqrt(n)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    print(f"\n========== РЕЗУЛЬТАТЫ {i} ==========")
    print(f"MAE   DQuant: {mae_dq:.6f}   GARCH: {mae_garch:.6f}")
    print(f"QLIKE DQuant: {qlike_dq:.6f}   GARCH: {qlike_garch:.6f}")
    print(f"DM-тест: статистика = {dm_stat:.4f}, p-value = {p_value:.4f}")
    if p_value < 0.05:
        print("=> Различие статистически значимо (на уровне 5%).")
    else:
        print("=> Различие не является статистически значимым.")


[*********************100%***********************]  1 of 1 completed

Загрузка SPY с 2014-01-01 по 2024-12-31...
Всего свечей: 2766
Период: 2014-01-03 – 2024-12-30

Train: 2265 свечей (до 2023-01-01)
Test:  501 свечей (с 2023-01-01)
                  open        high         low       close     volume  \
Date                                                                    
2014-01-03  148.832031  149.132579  148.344677  148.555862   81390600   
2014-01-06  149.043296  149.100148  147.897993  148.125427  108028200   
2014-01-07  148.718387  149.286973  148.604670  149.035172   86144200   
2014-01-08  149.010808  149.319474  148.555939  149.067673   96582300   
2014-01-09  149.546883  149.563132  148.482812  149.165115   90683400   
...                ...         ...         ...         ...        ...   
2022-12-23  363.950393  367.219383  362.397389  367.075592   59857300   
2022-12-27  366.960616  367.305715  363.950449  365.628082   51638200   
2022-12-28  365.560973  367.535814  360.854040  361.084106   70911500   
2022-12-29  363.931266  368.456082

Подготовка данных: |█████████████████████████████████████████████████░|  99.95%   Осталось 0.01 секундд
time spended:  15.655001163482666
[-0.00414399  0.00473453 -0.00721811 -0.00211063  0.00491803  0.00816101
  0.00660877  0.00334076 -0.00132457 -0.0118923  -0.01111316  0.00411439
  0.01069377 -0.02122985 -0.00904914  0.00784738  0.00686405  0.01042294
  0.00139604  0.00348094  0.00453452 -0.00234446  0.00202488 -0.00823246
  0.0031619   0.00464439  0.00297855  0.00010587 -0.00143415  0.00191241
 -0.00874204  0.00587185 -0.00106532  0.00149099  0.00968901  0.00089547
 -0.00474949 -0.0088187   0.00346271  0.00366195 -0.00632486  0.00838942
  0.00242922  0.00400108  0.00612763 -0.00073151  0.0051598   0.00160989
  0.00114095 -0.00051837  0.00202103  0.00650066  0.00477122  0.00102336
  0.00010213 -0.00348234 -0.00710534  0.00304402  0.00082367  0.00277534
  0.00731312  0.00112072  0.00202805 -0.0003066  -0.00604238  0.00450986
 -0.00071614  0.00194224 -0.00051048]
[0.00457561 0.0015834